In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Set Correct Paths**

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/content/sr_project")

HR_DIR = DATA_ROOT / "HR_256"
LR_DIR = DATA_ROOT / "LR_x4"

print("HR_DIR:", HR_DIR)
print("LR_DIR:", LR_DIR)

assert HR_DIR.exists(), "HR_256 folder not found!"
assert LR_DIR.exists(), "LR_x4 folder not found!"

HR_DIR: /content/drive/MyDrive/content/sr_project/HR_256
LR_DIR: /content/drive/MyDrive/content/sr_project/LR_x4


# **Define Train/Test IDs**

In [ ]:
def make_ids(start, end):
    return [f"{i:04d}" for i in range(start, end+1)]

train_ids = make_ids(0, 99)
test_ids  = make_ids(100, 109)

print("Train:", train_ids[:3], "...", train_ids[-3:])
print("Test:", test_ids)

Train: ['0000', '0001', '0002'] ... ['0097', '0098', '0099']
Test: ['0100', '0101', '0102', '0103', '0104', '0105', '0106', '0107', '0108', '0109']


# **Dataset Class (Patch Training)**

In [ ]:
from pathlib import Path
from PIL import Image
import torchvision.transforms.functional as TF
import random
import torch
from torch.utils.data import Dataset

def find_img(folder: Path, id_: str):
    for ext in [".png", ".jpg", ".jpeg"]:
        p = folder / f"{id_}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"Missing file for id={id_} in {folder}")

class SRDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, ids, scale=4, patch_size=24, augment=True):
        self.lr_dir = Path(lr_dir)
        self.hr_dir = Path(hr_dir)
        self.ids = ids
        self.scale = scale
        self.patch_size = patch_size
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id_ = self.ids[idx]

        lr_path = find_img(self.lr_dir, id_)
        hr_path = find_img(self.hr_dir, id_)

        lr = TF.to_tensor(Image.open(lr_path).convert("RGB"))
        hr = TF.to_tensor(Image.open(hr_path).convert("RGB"))

        _, h, w = lr.shape
        p = self.patch_size
        s = self.scale

        top = random.randint(0, h - p)
        left = random.randint(0, w - p)

        lr_patch = lr[:, top:top+p, left:left+p]
        hr_patch = hr[:, top*s:(top+p)*s, left*s:(left+p)*s]

        if self.augment:
            if random.random() < 0.5:
                lr_patch = TF.hflip(lr_patch)
                hr_patch = TF.hflip(hr_patch)
            if random.random() < 0.5:
                lr_patch = TF.vflip(lr_patch)
                hr_patch = TF.vflip(hr_patch)

        return lr_patch, hr_patch

# **Dataloaders**

In [ ]:
train_dataset = SRDataset(LR_DIR, HR_DIR, train_ids, patch_size=24)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print("Train batches:", len(train_loader))

Train batches: 4
